# 02 — Misclassification Study

Understand failure modes by visualizing misclassified light curves alongside
correctly classified examples. Ask: are the confusions astrophysically sensible?

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG
from src.data import get_object_lightcurve
from src.utils import plot_lightcurve, get_class_name, encode_labels, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
best_data = np.load('../../data/rnn_test_preds.npz', allow_pickle=True)
y_true = best_data['y_true']
y_pred = best_data['y_pred']
class_names = list(best_data['class_names'])

metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
lc = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet'))

# Reconstruct test set object IDs
# We need to re-run the same split to get object IDs
from src.preprocessing import batch_prepare_rnn
from sklearn.model_selection import train_test_split
from config import MODEL_CONFIG

all_targets = metadata['target'].values
_, encode_map, decode_map = encode_labels(all_targets)
labels = np.array([encode_map[t] for t in all_targets])

# Reproduce the split
rs = MODEL_CONFIG['random_state']
idx = np.arange(len(labels))
idx_trainval, idx_test = train_test_split(idx, test_size=MODEL_CONFIG['test_size'],
                                          stratify=labels, random_state=rs)
test_object_ids = metadata.iloc[idx_test]['object_id'].values
print(f"Test set: {len(idx_test)} objects")

## 1. Misclassified Examples by Class

For each class, plot misclassified light curves alongside correctly classified ones.

In [ ]:
# Find misclassified objects
wrong_mask = y_true != y_pred
wrong_indices = np.where(wrong_mask)[0]
print(f"Total misclassified: {len(wrong_indices)} / {len(y_true)} ({100*len(wrong_indices)/len(y_true):.1f}%)")

In [ ]:
# Plot misclassified vs correct for the top 4 most-confused pairs
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
n_classes = len(class_names)

# Get top confusion pairs
confusions = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            confusions.append((i, j, cm[i, j]))
confusions.sort(key=lambda x: -x[2])

fig, axes = plt.subplots(4, 2, figsize=(14, 16))

for row, (true_cls, pred_cls, count) in enumerate(confusions[:4]):
    # Find one misclassified example
    pair_mask = (y_true == true_cls) & (y_pred == pred_cls)
    mis_idx = np.where(pair_mask)[0]
    if len(mis_idx) == 0:
        continue

    # Plot misclassified
    obj_id = test_object_ids[mis_idx[0]]
    obj_lc = get_object_lightcurve(obj_id, lc)
    plot_lightcurve(obj_lc, object_id=obj_id,
                    title=f"MISCLASSIFIED: true={class_names[true_cls]}, pred={class_names[pred_cls]}",
                    ax=axes[row, 0], show_errors=False)

    # Plot a correct example of the true class
    correct_mask = (y_true == true_cls) & (y_pred == true_cls)
    correct_idx = np.where(correct_mask)[0]
    if len(correct_idx) > 0:
        correct_id = test_object_ids[correct_idx[0]]
        correct_lc = get_object_lightcurve(correct_id, lc)
        plot_lightcurve(correct_lc, object_id=correct_id,
                        title=f"CORRECT: {class_names[true_cls]}",
                        ax=axes[row, 1], show_errors=False)

plt.tight_layout()
save_figure(fig, 'misclassification_examples')
plt.show()

## 2. Are Confusions Astrophysically Sensible?

Some confusions are physically expected:
- **SN Iax <-> SN Ia**: Iax are subluminous Ia-like events, similar light curve shape
- **SN Ia-91bg <-> SN Ia**: 91bg are fast-declining Ia subtypes
- **SN II <-> SN Ibc**: Both are core-collapse SNe, similar timescales
- **AGN <-> long-duration transients**: Stochastic AGN variability can mimic slow transients

Confusions that would be concerning:
- **Periodic (RR Lyrae, EB) confused with transients**: These have fundamentally different morphologies
- **Microlensing confused with non-achromatic events**: Microlensing is achromatic (same in all bands)

Fill in observations after viewing the confusion matrix and examples above.

## 3. Confidence Analysis

In [ ]:
y_proba = best_data['y_pred_proba']

# Max predicted probability (confidence)
max_conf = y_proba.max(axis=1)
correct_mask = y_true == y_pred

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_conf[correct_mask], bins=50, alpha=0.6, label='Correct', density=True)
ax.hist(max_conf[~correct_mask], bins=50, alpha=0.6, label='Misclassified', density=True)
ax.set_xlabel('Max Predicted Probability (Confidence)')
ax.set_ylabel('Density')
ax.set_title('Model Confidence: Correct vs Misclassified')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean confidence (correct):       {max_conf[correct_mask].mean():.3f}")
print(f"Mean confidence (misclassified): {max_conf[~correct_mask].mean():.3f}")